In [1]:
import yfinance as yf
from datetime import datetime
import pandas as pd
# Set ticker symbols and date range
tickers = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA']
start_date = '2025-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

# Fetch and combine into one DataFrame
all_stocks = []

for ticker in tickers:
    df = yf.download(ticker, start=start_date, end=end_date)
    df = df.reset_index()
    df['Ticker'] = ticker
    all_stocks.append(df)

updated_stock_df = pd.concat(all_stocks, ignore_index=True)

# Format date
updated_stock_df['date'] = pd.to_datetime(updated_stock_df['Date']).dt.date

# Save if needed
# updated_stock_df.to_csv('updated_multi_stock.csv', index=False)

updated_stock_df.head()


/var/folders/60/w45408950l338hfzlxx878vc0000gn/T/ipykernel_23399/4062798555.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/var/folders/60/w45408950l338hfzlxx878vc0000gn/T/ipykernel_23399/4062798555.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/var/folders/60/w45408950l338hfzlxx878vc0000gn/T/ipykernel_23399/4062798555.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/var/folders/60/w45408950l338hfzlxx878vc0000gn/T/ipykernel_23399/4062798555.py:13: FutureWarning: YF.download() has changed argument 

Price,Date,Close,High,Low,Open,Volume,Ticker,Close,High,Low,...,High,Low,Open,Volume,Close,High,Low,Open,Volume,date
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL,,AMZN,AMZN,AMZN,...,MSFT,MSFT,MSFT,MSFT,TSLA,TSLA,TSLA,TSLA,TSLA,
0,2025-01-02,243.263199,248.500565,241.238085,248.330961,55740700.0,AAPL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-02
1,2025-01-03,242.774368,243.592387,241.307905,242.774368,40244100.0,AAPL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-03
2,2025-01-06,244.410416,246.734810,242.614744,243.722074,45045600.0,AAPL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-06
3,2025-01-07,241.627136,244.959095,240.769205,242.395272,40856000.0,AAPL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-07
4,2025-01-08,242.115936,243.123515,239.472320,241.337815,37628900.0,AAPL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-08


In [ ]:
import pandas as pd

# Load datasets
sentiment_df = pd.read_csv('../data/finbert/news_with_sentiment.csv')
stock_df = pd.read_csv('../data/raw/multi_stock_merged.csv')

# Try converting published_date safely
sentiment_df['published_date'] = pd.to_datetime(sentiment_df['published_date'], errors='coerce')

# Drop any rows where date parsing failed
sentiment_df = sentiment_df.dropna(subset=['published_date'])

# Extract just date
sentiment_df['date'] = sentiment_df['published_date'].dt.date

# Convert stock date
stock_df['Date'] = pd.to_datetime(stock_df['Date'], errors='coerce')
stock_df = stock_df.dropna(subset=['Date'])
stock_df['date'] = stock_df['Date'].dt.date


In [ ]:
sentiment_df[sentiment_df['stock_symbol'] == 'AAPL'].shape




In [ ]:
sentiment_df[sentiment_df['stock_symbol'] == 'AAPL']['date'].value_counts().sort_index()


In [ ]:
# One-hot encode sentiment column into separate columns
sentiment_encoded = pd.get_dummies(sentiment_df['sentiment'])
sentiment_df = pd.concat([sentiment_df, sentiment_encoded], axis=1)

# Group by stock and date to count sentiments
daily_sentiment = sentiment_df.groupby(['stock_symbol', 'date'])[['Positive', 'Neutral', 'Negative']].sum().reset_index()

# Preview the result
# daily_sentiment.head()

daily_sentiment[daily_sentiment['stock_symbol'] == 'AAPL']



In [ ]:
stock_df[stock_df['Ticker'] == 'AAPL']['date'].sort_values().tail(10)


In [ ]:
aapl_stock_dates = set(stock_df[stock_df['Ticker'] == 'AAPL']['date'])
aapl_sentiment_dates = set(daily_sentiment[daily_sentiment['stock_symbol'] == 'AAPL']['date'])

print("Common dates:", sorted(aapl_stock_dates & aapl_sentiment_dates))
print("Only in sentiment:", sorted(aapl_sentiment_dates - aapl_stock_dates))
print("Only in stock:", sorted(aapl_stock_dates - aapl_sentiment_dates))


In [ ]:
# Sort for consistency
stock_df = stock_df.sort_values(by=['Ticker', 'date'])

# Create next day's close
stock_df['next_close'] = stock_df.groupby('Ticker')['Close'].shift(-1)

# Calculate percentage change
stock_df['pct_change'] = (stock_df['next_close'] - stock_df['Close']) / stock_df['Close']

# Label movement
def label_movement(pct, threshold=0.005):
    if pct > threshold:
        return 2  # Up
    elif pct < -threshold:
        return 0  # Down
    else:
        return 1  # Neutral

stock_df['movement'] = stock_df['pct_change'].apply(label_movement)

# Preview
stock_df[['Ticker', 'date', 'Close', 'next_close', 'pct_change', 'movement']].head()


In [ ]:
# Merge on stock ticker and date
merged_df = pd.merge(
    daily_sentiment,
    stock_df[['Ticker', 'date', 'movement']],
    left_on=['stock_symbol', 'date'],
    right_on=['Ticker', 'date'],
    how='inner'
)

# Drop unnecessary columns
merged_df.drop(columns=['Ticker'], inplace=True)

# Fill NaNs (if any sentiment type didn't appear on that day)
merged_df[['Positive', 'Neutral', 'Negative']] = merged_df[['Positive', 'Neutral', 'Negative']].fillna(0)

# Preview the final dataset
merged_df.head()


In [ ]:
import numpy as np

def create_sequences(df, sequence_length=1):
    X, y = [], []
    
    df = df.sort_values(by='date')
    
    for i in range(len(df) - sequence_length):
        seq_x = df.iloc[i:i+sequence_length][['Positive', 'Neutral', 'Negative']].values
        seq_y = df.iloc[i + sequence_length]['movement']
        X.append(seq_x)
        y.append(seq_y)
    
    return np.array(X), np.array(y)

# Apply to each stock
X_all, y_all = [], []

for stock, group in merged_df.groupby('stock_symbol', group_keys=False):
    X_stock, y_stock = create_sequences(group, sequence_length=1)
    
    if len(X_stock) > 0:
        X_all.append(X_stock)
        y_all.append(y_stock)

# Stack all results
X = np.vstack(X_all)
y = np.concatenate(y_all)

print(f"Final shape: X={X.shape}, y={y.shape}")


In [ ]:
from sklearn.model_selection import train_test_split

# Chronological split: 80% train, 20% test
split_index = int(0.8 * len(X))

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Train shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test shape: X={X_test.shape}, y={y_test.shape}")
